# 01 · Caso de negocio — Eco-Score

Antes de modelar, el proyecto necesita **una pregunta de negocio, una métrica que la refleje y un costo de equivocarse**.
Este notebook deja esas tres cosas escritas **con números** y su resumen está en `reports/00_caso.md`.

> Estado: **borrador**. Las cifras salen de `data/raw/prosperLoanData.csv`, que solo se lee y nunca se modifica.
> Objetivo `is_default` y métrica AUC-PR aprobados por el grupo el 30 oct. Detalle de los hallazgos: `reports/01_eda.md`.

## 1. La pregunta de negocio

**Pregunta de negocio:** ¿podemos anticipar qué préstamos personales terminarán en impago y saber si vivir en un estado
con alto riesgo de inundación o precipitación extrema agrega información a esa predicción, más allá del perfil financiero?

**¿Quién decide y con qué información?** El analista de riesgo de una plataforma de préstamos entre personas (como Prosper).
Con un puntaje de probabilidad de impago **prioriza qué solicitudes revisar** y ajusta condiciones (monto, plazo, tasa).
El analista conserva la decisión final; el modelo solo ordena la cola de trabajo.

**¿Qué pasa hoy, sin el modelo?** El riesgo se evalúa con el perfil financiero y no hay forma de saber si el riesgo
climático del estado aporta algo. El índice climático del grupo (`Precipitation_Risk_Index`) es una **hipótesis**:
este proyecto la pone a prueba en lugar de suponerla.

In [1]:
from pathlib import Path

import numpy as np
import pandas as pd

# El notebook vive en notebooks/: la raíz del repositorio es la carpeta de arriba
ROOT = Path.cwd().resolve().parent if Path.cwd().name == "notebooks" else Path.cwd().resolve()
RAW_PATH = ROOT / "data" / "raw" / "prosperLoanData.csv"

# Solo las columnas que este notebook necesita. Las LP_* se usan ÚNICAMENTE para estimar costos de negocio:
# se conocen después de otorgar el préstamo y nunca serán variables del modelo.
COLUMNS = ["ListingKey", "MemberKey", "LoanStatus", "BorrowerState", "LoanOriginalAmount",
           "LP_NetPrincipalLoss", "LP_InterestandFees"]
raw = pd.read_csv(RAW_PATH, sep=";", usecols=COLUMNS, low_memory=False)
print(f"Préstamos en el archivo original: {len(raw):,}")
raw["LoanStatus"].value_counts()

Préstamos en el archivo original: 113,937


LoanStatus
Current                   56576
Completed                 38074
Chargedoff                11992
Defaulted                  5018
Past Due (1-15 days)        806
Past Due (31-60 days)       363
Past Due (61-90 days)       313
Past Due (91-120 days)      304
Past Due (16-30 days)       265
FinalPaymentInProgress      205
Past Due (>120 days)         16
Cancelled                     5
Name: count, dtype: int64

### ¿Qué préstamos sirven para responder la pregunta?

Solo los **terminados**: en un préstamo `Current` o `Past Due` todavía no sabemos si se pagará. La población (aprobada el 30 oct):

1. `LoanStatus` es `Completed` (pagó), `Chargedoff` o `Defaulted` (impago).
2. Una fila por préstamo (`ListingKey` aparece repetido en 889 filas).
3. Con `BorrowerState` conocido (sin estado no hay índice climático).

El objetivo aprobado es `is_default`: 1 = impago, 0 = pagó.

In [2]:
FINISHED = ["Completed", "Chargedoff", "Defaulted"]
DEFAULT = ["Chargedoff", "Defaulted"]

loans = (raw[raw["LoanStatus"].isin(FINISHED)]
         .drop_duplicates(subset="ListingKey", keep="first")
         .dropna(subset=["BorrowerState"])
         .copy())
loans["is_default"] = loans["LoanStatus"].isin(DEFAULT).astype(int)

members = loans["MemberKey"].value_counts()
print(f"Préstamos terminados para modelar: {len(loans):,}")
print(f"Prestatarios distintos: {loans['MemberKey'].nunique():,} "
      f"({(members > 1).sum():,} con más de un préstamo -> la partición será por MemberKey)")

priors = loans["is_default"].value_counts(normalize=True).sort_index()
pd.DataFrame({"préstamos": loans["is_default"].value_counts().sort_index(), "proporción": priors.round(4)},
             ).rename(index={0: "pagó (0)", 1: "impago (1)"})

Préstamos terminados para modelar: 49,552
Prestatarios distintos: 40,452 (7,028 con más de un préstamo -> la partición será por MemberKey)


,préstamos,proporción
is_default,,
pagó (0),34172,0.6896
impago (1),15380,0.3104


## 2. Equivocarse no cuesta lo mismo

En lugar de inventar la proporción entre errores, la estimamos con lo que **realmente pasó** en los préstamos terminados:

- **Error más caro — no detectar un impago** (falso negativo): se presta a quien no paga. Costo = capital perdido,
  medido con `LP_NetPrincipalLoss` en los préstamos en impago.
- **Error menos caro — marcar como riesgoso a quien sí paga** (falso positivo): se rechaza un buen crédito. Costo =
  intereses y cargos que se dejan de ganar, medidos con `LP_InterestandFees` en los préstamos pagados.

Es una aproximación: `LP_InterestandFees` es lo que pagó el prestatario, no la ganancia neta del inversionista, y el
analista puede ajustar condiciones en vez de rechazar.

In [3]:
defaulted = loans[loans["is_default"] == 1]
paid = loans[loans["is_default"] == 0]

loss_default = defaulted["LP_NetPrincipalLoss"].mean()
lost_interest = paid["LP_InterestandFees"].mean()
ratio = loss_default / lost_interest
ratio_median = defaulted["LP_NetPrincipalLoss"].median() / paid["LP_InterestandFees"].median()

loss_share = defaulted["LP_NetPrincipalLoss"].sum() / defaulted["LoanOriginalAmount"].sum()

print(f"Pérdida media de capital por impago:        {loss_default:,.2f} USD "
      f"({loss_share:.1%} del monto prestado en los impagos)")
print(f"Intereses y cargos medios de un buen cliente: {lost_interest:,.2f} USD")
print(f"Proporción con medias:   {ratio:.2f} a 1")
print(f"Proporción con medianas: {ratio_median:.2f} a 1")

Pérdida media de capital por impago:        4,672.50 USD (72.2% del monto prestado en los impagos)
Intereses y cargos medios de un buen cliente: 1,154.30 USD
Proporción con medias:   4.05 a 1
Proporción con medianas: 4.15 a 1


**Error más caro:** no detectar un impago. Se pierde en promedio el 72 % del monto prestado.

**Error menos caro:** rechazar a quien sí paga. Se pierden sus intereses, y el daño es recuperable porque el analista revisa.

**Proporción entre los dos: 4 a 1**, redondeando la proporción medida con medias y medianas (celda anterior).
La matriz usa unidades relativas, no dólares.

In [4]:
CLASSES = ["pagó", "impago"]
# Filas = lo que realmente pasó, columnas = lo que predijo el modelo. Diagonal = 0 (acertar no cuesta).
COST_MATRIX = np.array([[0, 1],
                        [4, 0]], dtype=float)

pd.DataFrame(COST_MATRIX, index=[f"real {c}" for c in CLASSES], columns=[f"predice {c}" for c in CLASSES])

,predice pagó,predice impago
real pagó,0.0,1.0
real impago,4.0,0.0


## 3. ¿Contra qué nos comparamos? (línea base)

Costo esperado por préstamo de tres estrategias **sin modelo**, con las proporciones medidas arriba:

- **Siempre "pagó"** (la clase más frecuente: aprobar a todos).
- **Siempre "impago"** (rechazar a todos).
- **Azar según las proporciones.**

In [5]:
p = priors.to_numpy()  # [pagó, impago]

baselines = pd.DataFrame({
    "costo esperado por préstamo": [p @ COST_MATRIX[:, 0], p @ COST_MATRIX[:, 1], p @ COST_MATRIX @ p],
    "exactitud": [p[0], p[1], p @ p],
}, index=["siempre 'pagó' (aprobar a todos)", "siempre 'impago' (rechazar a todos)", "azar según proporciones"])

print(f"AUC-PR de un modelo sin información = proporción de impagos = {p[1]:.4f}")
print(f"Umbral de decisión que minimiza el costo con esta matriz: marcar impago si P(impago) > {1 / (1 + 4):.2f}")
baselines.round(4)

AUC-PR de un modelo sin información = proporción de impagos = 0.3104
Umbral de decisión que minimiza el costo con esta matriz: marcar impago si P(impago) > 0.20


,costo esperado por préstamo,exactitud
siempre 'pagó' (aprobar a todos),1.2415,0.6896
siempre 'impago' (rechazar a todos),0.6896,0.3104
azar según proporciones,1.0702,0.5719


**Lo que dice la línea base:** aprobar a todos acierta en el 69 % de los casos, pero es la estrategia **más cara**:
cuesta 1,24 por préstamo. Con costos 4 a 1, hasta rechazar a todos sale más barato (0,69), aunque así no hay negocio.
El modelo tiene que costar **menos de 0,69 por préstamo** para valer la pena.

**¿La exactitud (accuracy) sirve como métrica en tu caso?** No. "Nadie cae en impago" ya da 68,96 % de exactitud y es
justamente la estrategia más costosa. La exactitud premia ignorar la clase que más cuesta.

## 4. Cómo se medirá el éxito

- **Métrica principal (aprobada el 30 oct):** **AUC-PR de la clase impago**. Mide si los impagos quedan arriba en la lista
  que revisa el analista, sin depender de un umbral. Un modelo sin información obtiene ≈ 0,31.
- **Métricas de control:**
  - costo esperado por préstamo con la matriz 4 a 1 y el umbral elegido en validación;
  - recall de impago;
  - ROC-AUC;
  - Brier (calibración).
- **Prueba de la hipótesis climática:** AUC-PR **con y sin** `Precipitation_Risk_Index`, con validación cruzada por
  `MemberKey`. Si la diferencia no supera la variación entre pliegues, el clima no agrega información.
- **Qué resultado haría que el negocio diga «sirve»:** AUC-PR claramente por encima de 0,31 **y** costo esperado por
  préstamo menor que 0,69 (la mejor estrategia sin modelo).

## 5. Lo que podría salir mal

Primero, un vistazo a la hipótesis climática: tasa de impago en los 10 estados que tienen índice asignado.

In [6]:
CLIMATE_INDEX = {"FL": 94.2, "LA": 91.0, "TX": 83.5, "NC": 77.1, "GA": 70.4,
                 "NY": 56.8, "CA": 48.0, "OH": 40.2, "NV": 14.1, "UT": 10.5}  # matriz de eco_score_pd.ipynb

by_state = (loans[loans["BorrowerState"].isin(CLIMATE_INDEX)]
            .groupby("BorrowerState")
            .agg(préstamos=("is_default", "size"), impago_pct=("is_default", "mean")))
by_state["impago_pct"] = (by_state["impago_pct"] * 100).round(1)
by_state["índice_climático"] = by_state.index.map(CLIMATE_INDEX)
others = loans.loc[~loans["BorrowerState"].isin(CLIMATE_INDEX), "is_default"]
print(f"Estados sin índice (reciben 50,0 por defecto): {others.size:,} préstamos, impago {others.mean() * 100:.1f} %")
by_state.sort_values("índice_climático", ascending=False)

Estados sin índice (reciben 50,0 por defecto): 26,622 préstamos, impago 30.6 %

,préstamos,impago_pct,índice_climático
BorrowerState,,,
FL,3073,32.0,94.2
LA,348,35.3,91.0
TX,2751,33.5,83.5
NC,1487,31.5,77.1
GA,2783,37.0,70.4
NY,2515,23.1,56.8
CA,7261,31.7,48.0
OH,1804,31.7,40.2
NV,339,29.8,14.1


- **Fuga de información:**
  - Las columnas `LP_*`, `ClosedDate`, `LoanCurrentDaysDelinquent`, `LoanFirstDefaultedCycleNumber` y
    `LoanMonthsSinceOrigination` se conocen después de otorgar el préstamo. Aquí solo se usan para estimar costos;
    se excluyen del modelo.
  - La escala de 4 niveles del notebook anterior (`Target_Risk_Scale`) se construía con el índice climático, que
    también es una variable: no puede ser el objetivo.
- **Sesgos o equidad:**
  - El índice asigna riesgo por estado de residencia y 41 estados reciben un valor por defecto.
  - En la tabla anterior, la tasa de impago no sube con el índice (UT 10,5 → 28,3 %; FL 94,2 → 32,0 %; NY 56,8 → 23,1 %).
  - Usarlo sin probarlo castigaría a prestatarios solo por dónde viven.
- **Datos personales o confidenciales:**
  - No hay nombres ni documentos.
  - `MemberKey`, `ListingKey`, `LoanKey` y `GroupKey` son llaves seudónimas: se quitan de las variables y
    `MemberKey` solo se usa para partir.
- **Hasta dónde puede decidir el modelo:**
  - El modelo prioriza la revisión; la decisión de crédito la toma una persona.
  - Muestra asociación, no causalidad: no puede afirmar que el clima cause impagos.
  - El índice es estatal y fijo, no un evento climático en la fecha del préstamo.
  - Sus valores son un **supuesto académico** del grupo, sin fuente verificable: no se presentan como dato oficial.

## 6. Cierre

- [x] Pregunta, quién decide y costos escritos **con números** (proporción 4 a 1 medida con los datos).
- [x] Línea base calculada (aprobar a todos: 1,24; rechazar a todos: 0,69; azar: 1,07).
- [x] Métrica principal elegida y justificada (AUC-PR, aprobada el 30 oct).
- [x] El grupo aprobó el objetivo `is_default` (préstamos terminados), el 30 oct.
- [x] Resumen copiado a `reports/00_caso.md`.
- [ ] Alguien que **no** escribió este notebook sabe explicar por qué un error cuesta más que el otro.